In [ ]:
EXP_NAME = "21MAY_finetuning_B"
data = "heart_disease_train"
param_space_path = "finetuning/param_space"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)

search_results = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/search_results.csv')
dataset = pd.read_csv(f'{PROJECT_ROOT}/data/{data}.csv')

with open(f"{PROJECT_ROOT}/configs/{param_space_path}.json", 'r') as f:
  param_space = json.load(f)

In [ ]:

search_results.head()

# Analysis

In [ ]:
search_results['delta_s_bal_acc'] = abs(search_results['mean_x_s_bal_acc'] - 0.5) - abs(search_results['mean_u_s_bal_acc'] - 0.5)
search_results['total_recon_loss'] = search_results['y_recon_loss'] + search_results['desc_recon_loss']

## Correlation analysis

In [ ]:
params = list(param_space.keys())
metrics = ["delta_s_bal_acc", "y_recon_loss", "desc_recon_loss", "total_recon_loss"]

In [ ]:
long_results = search_results[params + metrics].melt(id_vars=metrics, value_vars=params, value_name="param_value", var_name="parameter")

print(long_results.groupby(['parameter'])["param_value"].value_counts().sort_index())

In [ ]:
corr_matrix = search_results[params + metrics].corr(method='spearman')
param_metrics_corr = corr_matrix.loc[params, metrics]

plt.figure(figsize=(10, 6))
sns.heatmap(param_metrics_corr, annot=True, cmap='RdBu_r', center=0)
plt.title('Correlation: Scaling Params vs. Perf Metrics')
plt.show()

In [ ]:
metrics_metrics_corr = corr_matrix.loc[metrics, metrics]

plt.figure(figsize=(10, 6))
sns.heatmap(metrics_metrics_corr, annot=True, cmap='RdBu_r', center=0)
plt.title('Correlation: Perf Metrics')
plt.tight_layout()
plt.show()

### Desc Recon Alpha vs Pred Alpha

In [ ]:
alpha_pivot_1 = search_results.pivot_table(columns="desc_a", index="pred_a", values='delta_s_bal_acc', aggfunc="mean")
alpha_pivot_2 = search_results.pivot_table(columns="desc_a", index="pred_a", values='y_recon_loss', aggfunc="mean")
alpha_pivot_3 = search_results.pivot_table(columns="desc_a", index="pred_a", values='desc_recon_loss', aggfunc="mean")

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

sns.heatmap(alpha_pivot_1, annot=True, fmt=".3f", cmap="RdYlGn", ax=axes[0])
axes[0].set_title("Mean X->S vs U->S Balanced Accuracy Delta")
axes[0].set_xlabel("Desc Recon Alpha")
axes[0].set_ylabel("Pred Alpha")
sns.heatmap(alpha_pivot_2, annot=True, fmt=".3f", cmap="RdYlGn_r", ax=axes[1])
axes[1].set_title("Mean Y Prediction Loss")
axes[1].set_xlabel("Desc Recon Alpha")
axes[1].set_ylabel("Pred Alpha")
sns.heatmap(alpha_pivot_3, annot=True, fmt=".3f", cmap="RdYlGn_r", ax=axes[2])
axes[2].set_title("Mean Xdesc Reconstruction Loss")
axes[2].set_xlabel("Desc Recon Alpha")
axes[2].set_ylabel("Pred Alpha")

plt.tight_layout()
plt.show()

### CF Invariance Beta vs Total Correlation Beta

In [ ]:
beta_pivot_1 = search_results.pivot_table(columns="cf_invar_b", index="tc_b", values='delta_s_bal_acc', aggfunc="mean")
beta_pivot_2 = search_results.pivot_table(columns="cf_invar_b", index="tc_b", values='y_recon_loss', aggfunc="mean")
beta_pivot_3 = search_results.pivot_table(columns="cf_invar_b", index="tc_b", values='desc_recon_loss', aggfunc="mean")

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

sns.heatmap(beta_pivot_1, annot=True, fmt=".3f", cmap="RdYlGn", ax=axes[0])
axes[0].set_title("Mean X->S vs U->S Balanced Accuracy Delta")
axes[0].set_xlabel("CF Invar Beta")
axes[0].set_ylabel("TC Beta")
sns.heatmap(beta_pivot_2, annot=True, fmt=".3f", cmap="RdYlGn_r", ax=axes[1])
axes[1].set_title("Mean Y Prediction Loss")
axes[1].set_xlabel("CF Invar Beta")
axes[1].set_ylabel("TC Beta")
sns.heatmap(beta_pivot_3, annot=True, fmt=".3f", cmap="RdYlGn_r", ax=axes[2])
axes[2].set_title("Mean Xdesc Reconstruction Loss")
axes[2].set_xlabel("CF Invar Beta")
axes[2].set_ylabel("TC Beta")

plt.tight_layout()
plt.show()

### TC Beta vs Pred Alpha

In [ ]:
tc_pred_pivot_1 = search_results.pivot_table(columns="tc_b", index="pred_a", values='delta_s_bal_acc', aggfunc="mean")
tc_pred_pivot_2 = search_results.pivot_table(columns="tc_b", index="pred_a", values='y_recon_loss', aggfunc="mean")
tc_pred_pivot_3 = search_results.pivot_table(columns="tc_b", index="pred_a", values='desc_recon_loss', aggfunc="mean")

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

sns.heatmap(tc_pred_pivot_1, annot=True, fmt=".3f", cmap="RdYlGn", ax=axes[0])
axes[0].set_title("Mean X->S vs U->S Balanced Accuracy Delta")
axes[0].set_xlabel("TC Beta")
axes[0].set_ylabel("Pred Alpha")
sns.heatmap(tc_pred_pivot_2, annot=True, fmt=".3f", cmap="RdYlGn_r", ax=axes[1])
axes[1].set_title("Mean Y Prediction Loss")
axes[1].set_xlabel("TC Beta")
axes[1].set_ylabel("Pred Alpha")
sns.heatmap(tc_pred_pivot_3, annot=True, fmt=".3f", cmap="RdYlGn_r", ax=axes[2])
axes[2].set_title("Mean Xdesc Reconstruction Loss")
axes[2].set_xlabel("TC Beta")
axes[2].set_ylabel("Pred Alpha")

plt.tight_layout()
plt.show()

### CF Invar Beta vs Pred Alpha

In [ ]:
cf_invar_pred_pivot_1 = search_results.pivot_table(columns="cf_invar_b", index="pred_a", values='delta_s_bal_acc', aggfunc="mean")
cf_invar_pred_pivot_2 = search_results.pivot_table(columns="cf_invar_b", index="pred_a", values='y_recon_loss', aggfunc="mean")
cf_invar_pred_pivot_3 = search_results.pivot_table(columns="cf_invar_b", index="pred_a", values='desc_recon_loss', aggfunc="mean")

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

sns.heatmap(cf_invar_pred_pivot_1, annot=True, fmt=".3f", cmap="RdYlGn", ax=axes[0])
axes[0].set_title("Mean X->S vs U->S Balanced Accuracy Delta")
axes[0].set_xlabel("CF Invar Beta")
axes[0].set_ylabel("Pred Alpha")
sns.heatmap(cf_invar_pred_pivot_2, annot=True, fmt=".3f", cmap="RdYlGn_r", ax=axes[1])
axes[1].set_title("Mean Y Prediction Loss")
axes[1].set_xlabel("CF Invar Beta")
axes[1].set_ylabel("Pred Alpha")
sns.heatmap(cf_invar_pred_pivot_3, annot=True, fmt=".3f", cmap="RdYlGn_r", ax=axes[2])
axes[2].set_title("Mean Xdesc Reconstruction Loss")
axes[2].set_xlabel("CF Invar Beta")
axes[2].set_ylabel("Pred Alpha")

plt.tight_layout()
plt.show()

## Total Recon Loss vs S-Accuracy Delta Pareto Frontier 

**Objectives:** 
- MIMIMISE y_recon_loss and desc_recon_loss (`total_recon_loss`)
- MAXIMISE X->S vs U->S Balanced Accuracy Delta(`delta_s_bal_acc`)

In [ ]:
search_results['iteration'] = search_results['run'].str.split('_', expand=True)[0]

grouped_search_results = search_results.groupby('iteration')[params + metrics].agg('mean').sort_values('total_recon_loss')

pareto_frontier = []
current_max_delta_s_bal_acc = -0.5

print('--- Configurations on the Pareto Frontier ---')
for idx, solution in grouped_search_results.iterrows():
  if solution['delta_s_bal_acc'] > current_max_delta_s_bal_acc:
    pareto_frontier.append(solution[params + metrics])
    current_max_delta_s_bal_acc = solution['delta_s_bal_acc']
pareto_frontier_df = pd.DataFrame(pareto_frontier).sort_values('total_recon_loss')
print(pareto_frontier_df.to_markdown())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# OBJECTIVES = minimise TE error and minimise sensitive utility delta
xmax = max(grouped_search_results['delta_s_bal_acc'].max(), 0.01)
plt.axvspan(xmin=0, xmax=xmax, color="#b5ffc7", label='Ideal Udesc-Xdesc Sensitive Utility Delta', alpha=0.5)
sns.lineplot(data=pareto_frontier_df, x='delta_s_bal_acc', y='total_recon_loss', color="green", marker='', linestyle="--", label="Pareto Frontier", errorbar=None, ax=ax)
sns.scatterplot(data=grouped_search_results, x='delta_s_bal_acc', y='total_recon_loss', color="#a12aa5", alpha=0.8, sizes=(10, 400), ax=ax)
plt.xlim((grouped_search_results['delta_s_bal_acc'].min() - 0.001, xmax))
plt.xlabel('Udesc-Xdesc Sensitive Utility Delta')
plt.ylabel('Mean Total Reconstruction Loss')
handles, labels = ax.get_legend_handles_labels()
leg = ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1, 1), edgecolor="white")
plt.show()

## Y Recon Loss vs S-Accuracy Delta Pareto Frontier 

**Objectives:** 
- MIMIMISE `y_recon_loss`
- MAXIMISE X->S vs U->S Balanced Accuracy Delta(`delta_s_bal_acc`)

In [ ]:
search_results['iteration'] = search_results['run'].str.split('_', expand=True)[0]

grouped_search_results = search_results.groupby('iteration')[params + metrics].agg('mean').sort_values('y_recon_loss')

pareto_frontier = []
current_max_delta_s_bal_acc = -0.5

print('--- Configurations on the Pareto Frontier ---')
for idx, solution in grouped_search_results.iterrows():
  if solution['delta_s_bal_acc'] > current_max_delta_s_bal_acc:
    pareto_frontier.append(solution[params + metrics])
    current_max_delta_s_bal_acc = solution['delta_s_bal_acc']
pareto_frontier_df = pd.DataFrame(pareto_frontier).sort_values('y_recon_loss')
print(pareto_frontier_df.to_markdown())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# OBJECTIVES = minimise TE error and minimise sensitive utility delta
xmax = max(grouped_search_results['delta_s_bal_acc'].max(), 0.005)
plt.axvspan(xmin=0, xmax=xmax, color="#b5ffc7", label='Ideal Udesc-Xdesc Sensitive Utility Delta', alpha=0.5)
sns.lineplot(data=pareto_frontier_df, x='delta_s_bal_acc', y='y_recon_loss', color="green", marker='', linestyle="--", label="Pareto Frontier", errorbar=None, ax=ax)
sns.scatterplot(data=grouped_search_results, x='delta_s_bal_acc', y='y_recon_loss', color="#a12aa5", alpha=0.8, sizes=(10, 400), ax=ax)
plt.xlim((grouped_search_results['delta_s_bal_acc'].min() - 0.001, xmax))
plt.xlabel('Udesc-Xdesc Sensitive Utility Delta')
plt.ylabel('Mean Prediction Loss')
handles, labels = ax.get_legend_handles_labels()
leg = ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1, 1), edgecolor="white")
plt.show()

# All configs

In [ ]:
# Full list of explored configs

print(grouped_search_results.sort_values(params).to_markdown())